In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")

### 1. Исследование данных и целевой переменной

In [ ]:
# 1. Загрузка датасета
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedHouseVal'] = california.target

print("Размер датасета:", df.shape)
display(df.head())

# 2. Изучение целевой переменной
plt.figure(figsize=(8, 5))
sns.histplot(df['MedHouseVal'], bins=50, kde=True, color='blue')
plt.title('Распределение медианной стоимости домов (MedHouseVal)\n1 единица = $100,000')
plt.xlabel('Стоимость (в сотнях тыс. $)')
plt.ylabel('Количество районов')
plt.show()

### 2.  Выбор признаков и их визуализация

In [ ]:
# Выбираем 2 понятных признака:
# MedInc - медианный доход населения в районе (в десятках тысяч долларов)
# AveRooms - среднее количество комнат в доме

selected_features = ['MedInc', 'AveRooms']

# 3. Визуализация связи признаков с целевой переменной
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# График для MedInc
sns.scatterplot(data=df, x='MedInc', y='MedHouseVal', alpha=0.3, ax=axes[0])
axes[0].set_title('Зависимость цены от дохода (MedInc)')
axes[0].set_xlabel('Медианный доход (в 10 тыс. $)')
axes[0].set_ylabel('Стоимость дома (в 100 тыс. $)')

# График для AveRooms (ограничим ось X до 15 комнат, чтобы убрать выбросы и сделать график понятным)
sns.scatterplot(data=df, x='AveRooms', y='MedHouseVal', alpha=0.3, ax=axes[1], color='green')
axes[1].set_xlim(0, 15)
axes[1].set_title('Зависимость цены от кол-ва комнат (AveRooms)')
axes[1].set_xlabel('Среднее количество комнат')
axes[1].set_ylabel('Стоимость дома (в 100 тыс. $)')

plt.tight_layout()
plt.show()

### 3. Обучение и оценка линейной модели

In [ ]:
# Подготовка данных (только 2 признака)
X = df[selected_features]
y = df['MedHouseVal']

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Обучение LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 2. Предсказание
lr_preds = lr_model.predict(X_test)

# 3. Расчет метрик
mae_lr = mean_absolute_error(y_test, lr_preds)
mse_lr = mean_squared_error(y_test, lr_preds)
r2_lr = r2_score(y_test, lr_preds)

print("=== Оценка LinearRegression ===")
print(f"MAE (Средняя абсолютная ошибка): {mae_lr:.4f} (в сотнях тыс. $)")
print(f"MSE (Средняя квадратичная ошибка): {mse_lr:.4f}")
print(f"R2 (Коэффициент детерминации): {r2_lr:.4f}")

### 4. Интерпретация коэффициентов

In [ ]:
# 4. Вывод коэффициентов
print("Свободный член (Intercept):", round(lr_model.intercept_, 4))
print("Коэффициенты (Coef):")
for feature, coef in zip(selected_features, lr_model.coef_):
    print(f" - {feature}: {round(coef, 4)}")

Интерпретация модели для риэлторского агентства:
- Intercept (Базовая стоимость): Если представить фантастическую ситуацию, где медианный доход района равен нулю и в доме 0 комнат, базовая математическая оценка составила бы ... (значение Intercept).
- Признак MedInc (Доход): Коэффициент при MedInc равен ~0.43. Это значит, что при увеличении медианного дохода в районе на 1 единицу (10,000), прогнозируемая стоимость дома вырастает в среднем на 43,000 (так как целевая переменная измеряется в сотнях тысяч).
- Признак AveRooms (Комнаты): Коэффициент при AveRooms равен ~-0.03. Это значит, что (при прочих равных) добавление одной комнаты немного снижает стоимость (примерно на $3,000). В реальности это объясняется тем, что многокомнатные дома часто находятся в отдаленных/дешевых районах, а дорогие дома в центре могут иметь меньше комнат.

### 5. Сравнение с нелинейной моделью

In [ ]:
# 1. Обучение DecisionTreeRegressor
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

# Предсказание
tree_preds = tree_model.predict(X_test)

# 2. Метрики для дерева
mae_tree = mean_absolute_error(y_test, tree_preds)
r2_tree = r2_score(y_test, tree_preds)

print("=== Оценка DecisionTreeRegressor (max_depth=3) ===")
print(f"MAE: {mae_tree:.4f}")
print(f"R2: {r2_tree:.4f}\n")

# 3. Сравнение
print("=== Сравнение моделей ===")
print(f"LinearRegression R2: {r2_lr:.4f} | MAE: {mae_lr:.4f}")
print(f"DecisionTree R2:     {r2_tree:.4f} | MAE: {mae_tree:.4f}")

Сравнение моделей:
По метрике R^2 и MAE мы видим, что Линейная регрессия справляется лучше (ее R^2 выше, а ошибка MAE ниже), чем Дерево решений с ограничением глубины (max_depth=3). Линейная модель смогла уловить сильную прямую зависимость между ценой и доходом (MedInc). Дереву с глубиной 3 не хватает сложности, чтобы точно смоделировать эту зависимость на непрерывных данных. Поэтому для визуализации мы выберем Линейную регрессию.

### 5. Визуализация предсказаний 

In [ ]:
# 4. Scatter plot (Реальность против Предсказания)
plt.figure(figsize=(8, 6))

# Рисуем точки: по оси X - реальная цена, по оси Y - предсказанная цена (используем Линейную модель как лучшую)
plt.scatter(y_test, lr_preds, alpha=0.3, color='purple', label='Предсказания модели')

# Идеальная прямая y = x
min_val = min(y_test.min(), lr_preds.min())
max_val = max(y_test.max(), lr_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Идеальное предсказание (y=x)')

plt.title('Реальная стоимость vs Предсказанная стоимость (Linear Regression)')
plt.xlabel('Реальная стоимость дома (y_test)')
plt.ylabel('Предсказанная стоимость (lr_preds)')
plt.legend()
plt.show()

Анализ графика "Реальность vs Предсказание":
В идеале все фиолетовые точки должны были бы выстроиться точно на красной пунктирной линии (y=x).
На нашем графике видно, что:
1. Модель улавливает общую тенденцию (облако точек вытянуто вдоль красной линии).
2. Разброс точек достаточно широкий (отклонения от линии и есть наша ошибка MAE).
3. На правой границе (значение 5.0 на оси X) видна вертикальная линия точек. Это особенность датасета California Housing: все цены на дома выше $500,000 были искусственно обрезаны и записаны как 5.0. Наша модель не знает об этом искусственном потолке и пытается предсказывать значения и выше, и ниже 5.0.